# Film Festival Analytics: Data Science Interview Simulation

Your Name

**Duration: ~60 minutes | Difficulty: Intermediate**

## Getting Started

* Edit Your Name
* Automatic 0 if you include your student id or any other id
* Name the file Your_Name.ipynb
* Share the link with Anyone with link can View privileges
* Paste the shared link in Canvas

## How to Use This Notebook

This notebook is designed to be accessible to all users, including those navigating with a **screen reader**.

### Screen Reader Navigation Guide

Every code section in this notebook gives you a choice. Before each code cell you will find a navigation landmark containing two options:

- **Skip this code cell** — Jumps past the code directly to the output explanation. Use this if you want to focus on understanding results without reading raw code.
- **Go back and read the code cell** — Located after each explanation, this returns you to the top of the code section if you want to review it.

Each code cell is preceded by a simple description of what the code does, and followed by a description of the expected output. You can fully engage with this notebook without reading a single line of code if you prefer.

**Tips for screen reader users:**
- Press **H** to jump between major section headings
- Press **K** or **Tab** to move between links, including the skip and return navigation links
- Press **D** to jump between landmark regions (each navigation zone is a landmark)
- The skip and return links are inside labeled navigation landmarks — your screen reader will announce the landmark name so you always know where you are

---

## Learning Goals

By the end of this assignment you will be able to:

1. Write multi-table SQL JOIN queries to answer real business questions
2. Use aggregate functions (COUNT, SUM, AVG) with GROUP BY and HAVING
3. Load SQL query results into pandas DataFrames for further analysis
4. Perform time-series analysis and audience segmentation in Python
5. Design a relational database schema with appropriate keys and constraints
6. Identify and diagnose common data quality issues

---

## Deliverables

Complete each of the following and save your notebook before submission:

| Task | Description | Skills Tested |
|------|-------------|---------------|
| 1.1 | List all screening series | Basic SELECT |
| 1.2 | Films by Drama or Thriller directors | JOIN, WHERE IN |
| 2.1 | Box office revenue summary | Aggregation, ORDER BY |
| 2.2 | High-value audience members | HAVING, multi-condition filter |
| 3.1 | Film budget-to-attendance ratio | Complex JOIN, subqueries |
| 3.2 | Most-attended director | 5-table JOIN |
| 4.1 | Audience segmentation by membership tier | pandas GroupBy |
| 4.2 | Daily ticket trend and rolling average | pandas datetime |
| 5.1 | Schema design for 3 new features | Data modeling |
| 5.2 | Written design justification | Communication |
| 6.1 | Data quality checks | Diagnostic SQL |
| Bonus | Revenue forecasting for a new series | Open-ended analysis |

---

## Instructions

- Read each scenario carefully before writing any code
- Write your SQL inside the provided string variables
- Uncomment the `pd.read_sql_query(...)` lines to test your queries
- After each code cell, read the explanation cell to confirm your output matches expectations
- For written tasks (Task 5.2, Reflection), type directly into the Markdown cell provided
- **Do not delete existing code** — add your work where you see `# YOUR CODE HERE`

---

## Setup: Film Festival Database

You are interviewing for a Data Analyst position at a regional film festival organization that runs screening series throughout the year, tracking directors, films, audience members, and ticket sales.

**Run Setup Cells 1 through 3 in order before attempting any problems.**

<a name="read-code-1"></a>

**Setup Cell 1 of 3 — Import Libraries and Connect to Database**

This cell imports four Python libraries: `pandas` for data manipulation, `numpy` for numerical operations, `sqlite3` for local database connectivity, and `datetime` utilities for working with dates. It then opens a connection to a local SQLite file called `festival_interview.db` and creates a cursor object used to send SQL commands to the database.

<nav aria-label="Code cell 1 navigation">
<a href="#skip-code-1" aria-label="Skip code cell 1 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime, timedelta

# Create database connection
conn = sqlite3.connect('festival_interview.db')
cursor = conn.cursor()

print("Database connection established")

<a name="skip-code-1"></a>

**Expected output:** `Database connection established`

If you see an error, confirm you are running Python 3. sqlite3 is included in the Python standard library and requires no separate installation.

<nav aria-label="Return navigation for code cell 1">
<a href="#read-code-1" aria-label="Go back and read code cell 1">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-2"></a>

**Setup Cell 2 of 3 — Create the Database Schema**

This cell creates six tables: `directors`, `films`, `screening_series`, `series_films` (a junction table modeling the many-to-many relationship between series and films), `audience_members`, and `tickets`. The `IF NOT EXISTS` clause makes the cell safe to re-run. Foreign key declarations enforce referential integrity between related tables.

<nav aria-label="Code cell 2 navigation">
<a href="#skip-code-2" aria-label="Skip code cell 2 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# Create the film festival database schema

cursor.execute('''
CREATE TABLE IF NOT EXISTS directors (
    director_id INTEGER PRIMARY KEY,
    director_name TEXT NOT NULL,
    birth_year INTEGER,
    nationality TEXT,
    genre_specialty TEXT
)
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS films (
    film_id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    director_id INTEGER,
    release_year INTEGER,
    runtime_minutes INTEGER,
    budget_usd DECIMAL(12, 2),
    FOREIGN KEY (director_id) REFERENCES directors(director_id)
)
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS screening_series (
    series_id INTEGER PRIMARY KEY,
    series_name TEXT NOT NULL,
    start_date DATE,
    end_date DATE,
    venue TEXT,
    ticket_price DECIMAL(6, 2)
)
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS series_films (
    series_id INTEGER,
    film_id INTEGER,
    PRIMARY KEY (series_id, film_id),
    FOREIGN KEY (series_id) REFERENCES screening_series(series_id),
    FOREIGN KEY (film_id) REFERENCES films(film_id)
)
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS audience_members (
    member_id INTEGER PRIMARY KEY,
    first_name TEXT,
    last_name TEXT,
    email TEXT,
    membership_tier TEXT
)
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS tickets (
    ticket_id INTEGER PRIMARY KEY,
    member_id INTEGER,
    series_id INTEGER,
    attendance_date DATE,
    price_paid DECIMAL(6, 2),
    session_minutes INTEGER,
    FOREIGN KEY (member_id) REFERENCES audience_members(member_id),
    FOREIGN KEY (series_id) REFERENCES screening_series(series_id)
)
''')

conn.commit()
print("Schema created successfully")

<a name="skip-code-2"></a>

**Expected output:** `Schema created successfully`

Six tables now exist in the database. Notice the `series_films` table uses a composite primary key — the combination of `series_id` and `film_id` together must be unique. This is the standard pattern for modeling many-to-many relationships in relational databases.

<nav aria-label="Return navigation for code cell 2">
<a href="#read-code-2" aria-label="Go back and read code cell 2">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-3"></a>

**Setup Cell 3 of 3 — Insert Sample Data**

This cell populates all six tables with realistic film festival data: 7 directors spanning genres from Drama to Science Fiction, 12 films with production budgets, 4 screening series with dates and ticket prices, 8 audience members with different membership tiers, and 12 ticket purchase records. `executemany()` inserts multiple rows efficiently. `INSERT OR IGNORE` skips rows that would violate a primary key constraint, making the cell safe to re-run.

<nav aria-label="Code cell 3 navigation">
<a href="#skip-code-3" aria-label="Skip code cell 3 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# Insert sample data

directors_data = [
    (1, 'Akira Kurosawa', 1910, 'Japanese', 'Drama'),
    (2, 'Alfred Hitchcock', 1899, 'British', 'Thriller'),
    (3, 'Ingmar Bergman', 1918, 'Swedish', 'Drama'),
    (4, 'Agnes Varda', 1928, 'French', 'Documentary'),
    (5, 'Stanley Kubrick', 1928, 'American', 'Science Fiction'),
    (6, 'Wong Kar-wai', 1958, 'Hong Kong', 'Romance'),
    (7, 'Chantal Akerman', 1950, 'Belgian', 'Documentary')
]
cursor.executemany('INSERT OR IGNORE INTO directors VALUES (?, ?, ?, ?, ?)', directors_data)

films_data = [
    (1,  'Seven Samurai',          1, 1954, 207, 500000),
    (2,  'Rashomon',               1, 1950, 88,  260000),
    (3,  'Rear Window',            2, 1954, 112, 1000000),
    (4,  'Vertigo',                2, 1958, 128, 2479000),
    (5,  'The Seventh Seal',       3, 1957, 96,  150000),
    (6,  'Wild Strawberries',      3, 1957, 91,  200000),
    (7,  'Cleo from 5 to 7',       4, 1962, 90,  75000),
    (8,  'The Gleaners and I',     4, 2000, 82,  120000),
    (9,  '2001: A Space Odyssey',  5, 1968, 149, 10500000),
    (10, 'Full Metal Jacket',      5, 1987, 116, 30000000),
    (11, 'In the Mood for Love',   6, 2000, 98,  6200000),
    (12, 'Jeanne Dielman',         7, 1975, 201, 120000)
]
cursor.executemany('INSERT OR IGNORE INTO films VALUES (?, ?, ?, ?, ?, ?)', films_data)

series_data = [
    (1, 'Masters of World Cinema',  '2024-01-10', '2024-03-20', 'Main Theater',   18.00),
    (2, 'Hitchcock Retrospective',  '2024-02-01', '2024-03-31', 'Studio Hall',    22.00),
    (3, 'Visions of the Future',    '2024-03-01', '2024-05-15', 'IMAX Auditorium',28.00),
    (4, 'Documentary Spotlight',    '2024-04-01', '2024-05-31', 'Screening Room', 15.00)
]
cursor.executemany('INSERT OR IGNORE INTO screening_series VALUES (?, ?, ?, ?, ?, ?)', series_data)

series_films_data = [
    (1, 1), (1, 2), (1, 5), (1, 6),   # Masters of World Cinema
    (2, 3), (2, 4),                    # Hitchcock Retrospective
    (3, 9), (3, 10), (3, 11),          # Visions of the Future
    (4, 7), (4, 8), (4, 12)            # Documentary Spotlight
]
cursor.executemany('INSERT OR IGNORE INTO series_films VALUES (?, ?)', series_films_data)

members_data = [
    (1, 'Priya',   'Sharma',    'psharma@email.com',    'Patron'),
    (2, 'Carlos',  'Mendez',    'cmendez@email.com',    'Standard'),
    (3, 'Anika',   'Okonkwo',   'aokonkwo@email.com',   'Patron'),
    (4, 'Lena',    'Fischer',   'lfischer@email.com',   'Standard'),
    (5, 'Omar',    'Hassan',    'ohassan@email.com',    'Basic'),
    (6, 'Yuki',    'Tanaka',    'ytanaka@email.com',    'Patron'),
    (7, 'Sofia',   'Petrov',    'spetrov@email.com',    'Standard'),
    (8, 'Marcus',  'Williams',  'mwilliams@email.com',  'Basic')
]
cursor.executemany('INSERT OR IGNORE INTO audience_members VALUES (?, ?, ?, ?, ?)', members_data)

tickets_data = [
    (1,  1, 1, '2024-01-15', 18.00, 210),
    (2,  1, 2, '2024-02-10', 22.00, 130),
    (3,  2, 1, '2024-01-22', 18.00,  95),
    (4,  3, 1, '2024-02-05', 18.00, 180),
    (5,  3, 3, '2024-03-12', 28.00, 155),
    (6,  4, 2, '2024-02-18', 22.00,  90),
    (7,  5, 1, '2024-01-28', 18.00,  75),
    (8,  6, 3, '2024-03-20', 28.00, 160),
    (9,  7, 2, '2024-02-25', 22.00, 115),
    (10, 8, 1, '2024-02-08', 18.00,  80),
    (11, 2, 4, '2024-04-14', 15.00,  88),
    (12, 4, 4, '2024-04-22', 15.00,  92)
]
cursor.executemany('INSERT OR IGNORE INTO tickets VALUES (?, ?, ?, ?, ?, ?)', tickets_data)

conn.commit()
print("Sample data inserted successfully")
print(f"\nDirectors:       {len(directors_data)}")
print(f"Films:           {len(films_data)}")
print(f"Screening series:{len(series_data)}")
print(f"Members:         {len(members_data)}")
print(f"Tickets:         {len(tickets_data)}")

<a name="skip-code-3"></a>

**Expected output:** A summary showing 7 directors, 12 films, 4 screening series, 8 members, and 12 tickets.

The database is now fully populated and ready. Proceed to Problem 1.

**Database relationship summary:**
- Each film belongs to one director
- Each screening series can feature many films (and a film can appear in many series) via the series_films junction table
- Each ticket links one audience member to one screening series on a specific date

<nav aria-label="Return navigation for code cell 3">
<a href="#read-code-3" aria-label="Go back and read code cell 3">&#8617; Go back and read the code cell</a>
</nav>

---
## Problem 1: Data Exploration
**Estimated time: 10 minutes**

**Scenario:** The festival's new programming director wants a quick overview of what is currently scheduled.

**Skills practiced:** Basic SELECT queries, WHERE with IN, two-table JOIN, column aliasing

**Step-by-step approach:**
1. Identify which table(s) contain the data you need
2. Decide which columns to SELECT
3. Apply any filters using WHERE
4. Uncomment the `pd.read_sql_query` line to test your query

### Task 1.1: List All Screening Series

Write a query to show all screening series with their name, venue, date range, and ticket price, ordered by start date.

**Hint:** All the data you need is in the `screening_series` table.

<a name="read-code-4"></a>

**Task 1.1 Code Cell**

Your SQL query goes inside the `query_1_1` string between the triple quotes. SELECT the series name, venue, start date, end date, and ticket price from the `screening_series` table. Uncomment the last two lines to run the query and display results as a DataFrame.

<nav aria-label="Code cell 4 navigation">
<a href="#skip-code-4" aria-label="Skip code cell 4 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
query_1_1 = """
-- Write your SQL query here


"""

# Test your query
# df_series = pd.read_sql_query(query_1_1, conn)
# print(df_series)

<a name="skip-code-4"></a>

**Expected result:** A table with 4 rows — one per screening series — showing names, venues, date ranges, and ticket prices ranging from $15.00 to $28.00, ordered chronologically by start date.

**Common mistakes to watch for:**
- Using `SELECT *` works but is considered poor practice — always name your columns explicitly
- If you get an error, confirm that Setup Cells 1 through 3 ran successfully first

<nav aria-label="Return navigation for code cell 4">
<a href="#read-code-4" aria-label="Go back and read code cell 4">&#8617; Go back and read the code cell</a>
</nav>

### Task 1.2: Find Films by Drama or Thriller Directors

Write a query returning all films directed by someone whose genre specialty is Drama or Thriller. Include the director name, film title, release year, and runtime.

**Hint:** JOIN `films` to `directors` on `director_id`, then filter with `WHERE genre_specialty IN ('Drama', 'Thriller')`

<a name="read-code-5"></a>

**Task 1.2 Code Cell**

Your query needs data from two tables: `directors` (for genre specialty and director name) and `films` (for title, release year, and runtime). Join them on the shared `director_id` column. Then filter to keep only Drama and Thriller directors.

<nav aria-label="Code cell 5 navigation">
<a href="#skip-code-5" aria-label="Skip code cell 5 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
query_1_2 = """
-- Write your SQL query here


"""

# Test your query
# df_drama_thriller = pd.read_sql_query(query_1_2, conn)
# print(df_drama_thriller)

<a name="skip-code-5"></a>

**Expected result:** 6 films — Seven Samurai and Rashomon by Kurosawa (Drama), The Seventh Seal and Wild Strawberries by Bergman (Drama), and Rear Window and Vertigo by Hitchcock (Thriller).

**Key concept:** `WHERE genre_specialty IN ('Drama', 'Thriller')` is more readable than chaining two OR conditions and scales easily if you need to add more genres.

<nav aria-label="Return navigation for code cell 5">
<a href="#read-code-5" aria-label="Go back and read code cell 5">&#8617; Go back and read the code cell</a>
</nav>

---
## Problem 2: Box Office Analysis
**Estimated time: 15 minutes**

**Scenario:** The festival's finance team wants to understand which screening series generate the most revenue and which audience members are most valuable.

**Skills practiced:** Aggregate functions (SUM, COUNT, AVG), GROUP BY, HAVING, ORDER BY, multi-table JOIN

**Step-by-step approach:**
1. Identify which tables to JOIN
2. Decide what to GROUP BY
3. Apply aggregate functions to each group
4. Use HAVING (not WHERE) to filter on aggregated values
5. Sort results meaningfully with ORDER BY

### Task 2.1: Screening Series Revenue Summary

Calculate total box office revenue, total ticket count, and average session length for each screening series. Order results by total revenue from highest to lowest.

**Hint:** JOIN `tickets` to `screening_series` on `series_id`. Use COUNT, SUM, and AVG. Group by series name.

<a name="read-code-6"></a>

**Task 2.1 Code Cell**

Your query should JOIN `tickets` and `screening_series` and group by series name. You need four output columns: series name, total ticket count (COUNT of ticket_id), total revenue (SUM of price_paid), and average session length (AVG of session_minutes). Sort so the highest-revenue series appears first.

<nav aria-label="Code cell 6 navigation">
<a href="#skip-code-6" aria-label="Skip code cell 6 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
query_2_1 = """
-- Write your SQL query here


"""

# Test your query
# df_revenue = pd.read_sql_query(query_2_1, conn)
# print(df_revenue)

<a name="skip-code-6"></a>

**Expected result:** 4 rows. Masters of World Cinema should have the most tickets sold (5). Visions of the Future should have the highest average session length (around 157 minutes).

**Interview follow-up:** The finance team will likely ask for a revenue-per-ticket comparison. Can you add that calculation to your query?

<nav aria-label="Return navigation for code cell 6">
<a href="#read-code-6" aria-label="Go back and read code cell 6">&#8617; Go back and read the code cell</a>
</nav>

### Task 2.2: Identify High-Value Audience Members

Find audience members who have purchased more than 1 ticket AND spent more than $35 total. Show their full name, membership tier, total tickets, and total amount spent.

**Hint:** Use HAVING — not WHERE — to filter on aggregated values. Join `tickets` and `audience_members`.

<a name="read-code-7"></a>

**Task 2.2 Code Cell**

Join `tickets` to `audience_members`, group by member, and apply two HAVING conditions: `COUNT(*) > 1` for ticket count and `SUM(price_paid) > 35` for total spending. Remember: WHERE filters individual rows before grouping; HAVING filters groups after aggregation.

<nav aria-label="Code cell 7 navigation">
<a href="#skip-code-7" aria-label="Skip code cell 7 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
query_2_2 = """
-- Write your SQL query here


"""

# Test your query
# df_high_value = pd.read_sql_query(query_2_2, conn)
# print(df_high_value)

<a name="skip-code-7"></a>

**Expected result:** 3 audience members meet both criteria: Priya Sharma (Patron, 2 tickets, \$40 total), Anika Okonkwo (Patron, 2 tickets, \$46 total), and Carlos Mendez (Standard, 2 tickets, \$33 total — check whether Carlos qualifies given the \$35 threshold).

**Common mistake:** Using WHERE instead of HAVING to filter on COUNT or SUM. WHERE runs before aggregation and cannot see group-level totals.

<nav aria-label="Return navigation for code cell 7">
<a href="#read-code-7" aria-label="Go back and read code cell 7">&#8617; Go back and read the code cell</a>
</nav>

---
## Problem 3: Complex Multi-Table Analysis
**Estimated time: 15 minutes**

**Scenario:** The programming team wants deeper insights to guide curation and pricing decisions.

**Skills practiced:** Multi-table JOINs (4 to 5 tables), nested aggregation, subqueries or CTEs

**Step-by-step approach:**
1. Sketch out the join path before writing SQL
2. Get the JOINs right first, then add aggregations
3. Use table aliases (e.g., `FROM tickets t`) to keep queries readable
4. Test intermediate steps by running partial queries

### Task 3.1: Film Budget-to-Attendance Ratio

For each screening series, calculate: total production budget of all films shown, number of unique attendees, and budget per attendee (total budget divided by attendee count).

**Join path:** screening_series → series_films → films (for budget), and screening_series → tickets (for attendees). Consider writing each aggregation as a subquery or CTE and joining them on series_id.

<a name="read-code-8"></a>

**Task 3.1 Code Cell**

This query involves two separate aggregations: one that sums film budgets (via series_films joined to films) and one that counts unique attendees (via tickets). Consider writing each as a subquery and joining them on series_id. Divide total budget by attendee count to compute the ratio.

<nav aria-label="Code cell 8 navigation">
<a href="#skip-code-8" aria-label="Skip code cell 8 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
query_3_1 = """
-- Write your SQL query here
-- Hint: You may need subqueries or a CTE (WITH clause)


"""

# Test your query
# df_budget_analysis = pd.read_sql_query(query_3_1, conn)
# print(df_budget_analysis)

<a name="skip-code-8"></a>

**Expected result:** 4 rows. Visions of the Future should show the highest total budget (combining 2001: A Space Odyssey at \$10.5M and Full Metal Jacket at \$30M) with a correspondingly high budget-per-attendee figure.

**Analytical insight:** A high budget-per-attendee ratio may indicate that a high-prestige series is under-attended relative to the caliber of its films — a programming opportunity.

<nav aria-label="Return navigation for code cell 8">
<a href="#read-code-8" aria-label="Go back and read code cell 8">&#8617; Go back and read the code cell</a>
</nav>

### Task 3.2: Most-Attended Director

Determine which director's films have drawn the most total attendees across all screening series. Show: director name, count of their films in series, total attendance, and average session length.

**Join path (5 tables):** directors → films → series_films → screening_series → tickets

<a name="read-code-9"></a>

**Task 3.2 Code Cell**

Trace the full chain: directors own films, films appear in series via series_films, and series accumulate ticket purchases. Join all five tables. Group by director name and use COUNT and AVG to aggregate the metrics.

<nav aria-label="Code cell 9 navigation">
<a href="#skip-code-9" aria-label="Skip code cell 9 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
query_3_2 = """
-- Write your SQL query here
-- Hint: Join directors -> films -> series_films -> screening_series -> tickets


"""

# Test your query
# df_popular_directors = pd.read_sql_query(query_3_2, conn)
# print(df_popular_directors)

<a name="skip-code-9"></a>

**Expected result:** Kurosawa and Bergman should appear with the most attendance, as both have films in the well-attended Masters of World Cinema series. Hitchcock's films anchor the dedicated Hitchcock Retrospective series.

**Interview tip:** Practice explaining your JOIN chain step by step: 'I start with directors, join to films on director_id, then to series_films on film_id, then to tickets on series_id.'

<nav aria-label="Return navigation for code cell 9">
<a href="#read-code-9" aria-label="Go back and read code cell 9">&#8617; Go back and read the code cell</a>
</nav>

---
## Problem 4: Python Data Analysis
**Estimated time: 15 minutes**

**Scenario:** The marketing team needs insights that require more complex analysis than SQL alone can provide.

**Skills practiced:** pandas GroupBy, DataFrame operations, datetime parsing, rolling averages

**Step-by-step approach:**
1. Load data from SQL into a pandas DataFrame
2. Inspect shape and data types before transforming
3. Use `.groupby()` followed by `.agg()` to compute multiple metrics at once
4. For time series: parse dates first, then compute rolling statistics

### Task 4.1: Audience Segmentation by Membership Tier

Using pandas, analyze audience behavior by membership tier. Calculate for each tier: average spending per ticket, average session length, number of unique members, and total revenue contribution.

<a name="read-code-10"></a>

**Task 4.1 Code Cell**

Starter SQL is provided to load the data into `df_tickets`. Your work begins at Step 2. Use `df_tickets.groupby('membership_tier')` followed by `.agg()` to compute multiple metrics. To count unique members within each group, use `.nunique()` on the member_id column.

<nav aria-label="Code cell 10 navigation">
<a href="#skip-code-10" aria-label="Skip code cell 10 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# Step 1: Load the data (provided for you)
query = """
SELECT
    t.ticket_id,
    t.member_id,
    a.membership_tier,
    t.price_paid,
    t.session_minutes
FROM tickets t
JOIN audience_members a ON t.member_id = a.member_id
"""

df_tickets = pd.read_sql_query(query, conn)

# Step 2: Perform your segmentation analysis
# YOUR CODE HERE
# Hint: df_tickets.groupby('membership_tier').agg({...})


# Step 3: Display results clearly
# YOUR CODE HERE

<a name="skip-code-10"></a>

**Expected result:** A summary table with 3 rows (Basic, Patron, Standard) showing metrics per tier. Patron members should show higher average session length, reflecting deeper engagement with the content.

**pandas tip:** `.agg()` accepts a dictionary where keys are column names and values are aggregation functions. Example: `{'price_paid': ['mean', 'sum'], 'member_id': 'nunique'}`

<nav aria-label="Return navigation for code cell 10">
<a href="#read-code-10" aria-label="Go back and read code cell 10">&#8617; Go back and read the code cell</a>
</nav>

### Task 4.2: Daily Ticket Trend Analysis

Create a time series showing daily ticket counts and a 7-day rolling average. Identify the busiest day of the week.

**Hint:** Use `pd.to_datetime()` to parse dates, `.rolling(7).mean()` for the rolling average, and `.dt.day_name()` to extract the day of the week.

<a name="read-code-11"></a>

**Task 4.2 Code Cell**

Starter SQL is provided to load daily ticket counts into `df_daily`. Step 1: Convert the `attendance_date` column from string to pandas datetime using `pd.to_datetime()`. Step 2: Compute a 7-day rolling average of `ticket_count` using `.rolling(window=7).mean()`. Step 3: Add a day-of-week column using `.dt.day_name()` and identify which day has the highest average count.

<nav aria-label="Code cell 11 navigation">
<a href="#skip-code-11" aria-label="Skip code cell 11 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# Load daily ticket counts (provided for you)
query = """
SELECT
    attendance_date,
    COUNT(*) as ticket_count
FROM tickets
GROUP BY attendance_date
ORDER BY attendance_date
"""

df_daily = pd.read_sql_query(query, conn)

# Step 1: Convert attendance_date to datetime
# YOUR CODE HERE

# Step 2: Compute 7-day rolling average
# YOUR CODE HERE

# Step 3: Identify the busiest day of the week
# YOUR CODE HERE

<a name="skip-code-11"></a>

**Expected result:** A DataFrame with dates, ticket counts, and a rolling average column. With only 12 tickets in the sample, the rolling average will be sparse — this analysis is most powerful with production-scale data covering months or years.

**Interview insight:** Be ready to discuss how you would visualize this and what programming decisions the trend line might inform — for example, scheduling high-demand series on peak attendance days.

<nav aria-label="Return navigation for code cell 11">
<a href="#read-code-11" aria-label="Go back and read code cell 11">&#8617; Go back and read the code cell</a>
</nav>

---
## Problem 5: Database Design Challenge
**Estimated time: 15 minutes**

**Scenario:** The festival wants to expand their platform with three new features.

**Skills practiced:** Schema design, data type selection, primary and foreign key design, constraint thinking

**Step-by-step approach:**
1. Identify the entities (nouns) and relationships (verbs) for each feature
2. Choose appropriate data types for each column
3. Decide which columns should be NOT NULL, UNIQUE, or have DEFAULT values
4. Consider what queries will commonly run and design to support them

### Task 5.1: Design Three New Tables

Write complete `CREATE TABLE` SQL statements for:

1. **Equipment Rentals** — Track which audience members rent audio description headsets, the rental fee, and whether the equipment has been returned
2. **Film Ratings** — Allow audience members to rate individual films they have seen (1 to 5 stars) with an optional written review
3. **Concessions** — Track items sold at the concession stand with prices, and link purchases to audience members

Include all primary keys, foreign keys, data types, and any appropriate constraints.

<a name="read-code-12"></a>

**Task 5.1 Code Cell**

Three string variables are provided as templates. Fill in the column definitions. Think carefully about: What is the primary key? What foreign keys link back to existing tables? Should any columns have DEFAULT values (e.g., `returned DEFAULT 0` for a boolean flag)? Should ratings be constrained to 1-5 using `CHECK (rating BETWEEN 1 AND 5)`? Should a member be allowed to rate the same film more than once?

<nav aria-label="Code cell 12 navigation">
<a href="#skip-code-12" aria-label="Skip code cell 12 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# YOUR CODE HERE

# Equipment Rentals table
equipment_schema = """
CREATE TABLE equipment_rentals (
    -- YOUR CODE HERE
);
"""

# Film Ratings table
ratings_schema = """
CREATE TABLE film_ratings (
    -- YOUR CODE HERE
);
"""

# Concessions tables (you may need multiple tables)
concessions_schema = """
CREATE TABLE concession_items (
    -- YOUR CODE HERE
);

CREATE TABLE concession_purchases (
    -- YOUR CODE HERE
);
"""

print("Equipment Rentals Schema:")
print(equipment_schema)
print("\nFilm Ratings Schema:")
print(ratings_schema)
print("\nConcessions Schema:")
print(concessions_schema)

<a name="skip-code-12"></a>

**Self-check questions:**

- Does each table have a primary key?
- Does each table referencing another table have a FOREIGN KEY constraint?
- Have you used CHECK constraints where values are bounded (e.g., 1-5 star ratings)?
- Have you considered a UNIQUE constraint on film_ratings to prevent duplicate reviews?

Move on to Task 5.2 to explain your design decisions in writing.

<nav aria-label="Return navigation for code cell 12">
<a href="#read-code-12" aria-label="Go back and read code cell 12">&#8617; Go back and read the code cell</a>
</nav>

### Task 5.2: Explain Your Design Decisions

Edit this Markdown cell directly and write your response below. Address all four points:

1. **Why did you choose specific data types?** (e.g., why INTEGER vs REAL for ratings, why DECIMAL vs FLOAT for money)
2. **How did you handle relationships between tables?** (foreign keys, junction tables if needed for many-to-many)
3. **What assumptions did you make?** (e.g., can a member rate the same film more than once? Can equipment be rented multiple times?)
4. **How does your design support common queries?** (e.g., find all unreturned equipment, calculate a film's average rating)

---

**YOUR EXPLANATION HERE:**

*(Edit this cell and write your response here)*

---

---
## Bonus Challenge: Revenue Forecasting
**Optional | Open-ended**

**Scenario:** The board wants to project revenue for a new 45-day documentary series before committing to the venue contract.

**Your challenge:**
1. Calculate the average daily revenue across all existing screening series
2. Identify any patterns by day of week or position in a series run (early vs late)
3. Project estimated revenue for a new 45-day series priced at $20 per ticket
4. Recommend whether to book the 120-seat Main Theater or the 60-seat Screening Room based on projected attendance

Use both SQL and Python. There is no single correct answer — focus on demonstrating clear analytical reasoning and stating your assumptions explicitly.

<a name="read-code-13"></a>

**Bonus Code Cell**

This is open-ended. Suggested starting point: write a SQL query that calculates daily revenue per series by dividing total revenue by the number of days the series ran. Then use Python to project forward for 45 days at $20 per ticket. Document every assumption with a comment — interviewers evaluate assumptions as carefully as calculations.

<nav aria-label="Code cell 13 navigation">
<a href="#skip-code-13" aria-label="Skip code cell 13 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# YOUR CODE HERE
# Suggested starting point:

# Step 1: Calculate average daily revenue per series using SQL
# query_bonus = """
# SELECT
#     series_name,
#     SUM(price_paid) AS total_revenue,
#     julianday(end_date) - julianday(start_date) AS duration_days,
#     SUM(price_paid) / (julianday(end_date) - julianday(start_date)) AS daily_revenue
# FROM tickets t
# JOIN screening_series s ON t.series_id = s.series_id
# GROUP BY s.series_id
# """

# Step 2: Project revenue for a new 45-day series at $20/ticket
# YOUR CODE HERE

# Step 3: Venue recommendation based on projected attendance
# YOUR CODE HERE

<a name="skip-code-13"></a>

**Approaching open-ended problems in interviews:**

Interviewers are usually less interested in a precise number and more interested in how you structure your thinking. Make assumptions explicit, quantify uncertainty, and proactively flag limitations.

**Strong answer pattern:** State assumptions → Show calculation → Validate against known data → Identify what would change the forecast

<nav aria-label="Return navigation for code cell 13">
<a href="#read-code-13" aria-label="Go back and read code cell 13">&#8617; Go back and read the code cell</a>
</nav>

---
## Problem 6: Data Quality and Cleaning
**Estimated time: 10 minutes**

**Scenario:** Before the quarterly board report, you need to verify the integrity of the ticket data.

**Skills practiced:** Diagnostic SQL, LEFT JOIN for finding missing relationships, GROUP BY with HAVING for finding duplicates

**Step-by-step approach:**
1. For price mismatches: JOIN tickets to screening_series and compare the two price columns
2. For orphaned records: use LEFT JOIN and filter WHERE the right-side column IS NULL
3. For duplicates: GROUP BY email and use HAVING COUNT(*) > 1

### Task 6.1: Write Four Data Quality Checks

Write SQL queries to find:

1. Tickets where `price_paid` does not match the series' listed `ticket_price`
2. Films that have no associated director
3. Screening series that have no films assigned to them
4. Audience members with duplicate email addresses

<a name="read-code-14"></a>

**Task 6.1 Code Cell**

Four query variables are provided — fill in each one. For checks 2 and 3 (finding missing relationships), the LEFT JOIN IS NULL pattern works well: join table A to table B, then filter WHERE the table B key column IS NULL — this returns rows in table A with no match in table B. For check 4, GROUP BY email and use HAVING COUNT(*) > 1 to surface duplicates.

<nav aria-label="Code cell 14 navigation">
<a href="#skip-code-14" aria-label="Skip code cell 14 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# 1. Price mismatches between ticket records and listed series prices
query_price_check = """
-- YOUR QUERY HERE

"""

# 2. Films without an associated director
query_orphaned_films = """
-- YOUR QUERY HERE

"""

# 3. Screening series with no films assigned
query_empty_series = """
-- YOUR QUERY HERE

"""

# 4. Duplicate audience member email addresses
query_duplicate_emails = """
-- YOUR QUERY HERE

"""

# Run your checks (uncomment to test each one)
# print("Price Mismatches:")
# print(pd.read_sql_query(query_price_check, conn))
# print("\nOrphaned Films:")
# print(pd.read_sql_query(query_orphaned_films, conn))
# print("\nEmpty Series:")
# print(pd.read_sql_query(query_empty_series, conn))
# print("\nDuplicate Emails:")
# print(pd.read_sql_query(query_duplicate_emails, conn))

<a name="skip-code-14"></a>

**Expected results with the current sample data:**

- Price mismatches: 0 rows — all ticket prices match the series list price
- Orphaned films: 0 rows — all films have a director
- Empty series: 0 rows — all series have at least one film
- Duplicate emails: 0 rows — all email addresses are unique

**Real-world note:** Zero issues is a valid and important result. In production, these checks run on a schedule and trigger alerts the moment any row appears.

<nav aria-label="Return navigation for code cell 14">
<a href="#read-code-14" aria-label="Go back and read code cell 14">&#8617; Go back and read the code cell</a>
</nav>

---
## Reflection Questions

After completing the exercises, review and be prepared to discuss these in your interview. They are common follow-up questions from interviewers.

**1. Performance Optimization**
- Which of your queries might run slowly with 10 million ticket records?
- What indexes would you create to speed up the most common queries?
- How would you rewrite a slow query using a CTE or window function?

**2. Scalability**
- How would you modify this schema to support multiple festival locations across different cities?
- What would you change if ticket records grew to 50 million rows?
- When would you consider moving from SQLite to a cloud data warehouse?

**3. Data Governance and Privacy**
- What personally identifiable information (PII) does this database contain?
- How would you implement a data retention policy for audience member records?
- What access controls would you put around the audience_members table?

**4. Business Intelligence**
- What are the top 3 KPIs you would put on the festival director's dashboard?
- What predictive models could add the most value to this dataset?
- How would you explain the budget-to-attendance ratio to a non-technical board member?

**5. Technical Trade-offs**
- Why use a relational database instead of NoSQL for this use case?
- When would you denormalize a table for performance?
- How would you handle two staff members updating the same ticket record simultaneously?

---

## Key Concepts Summary

**SQL skills demonstrated in this assignment:**
Multi-table JOINs across up to 5 tables, aggregate functions (COUNT, SUM, AVG) with GROUP BY, HAVING for post-aggregation filtering, LEFT JOIN for finding missing or orphaned records, and subqueries or CTEs for complex logic.

**Data modeling principles:**
Primary and foreign key design, junction tables for many-to-many relationships, appropriate data type selection, and constraints for data integrity (NOT NULL, UNIQUE, CHECK).

**Python and pandas:**
Loading SQL results into DataFrames, GroupBy with multiple aggregations, datetime parsing and rolling statistics, and communicating analytical findings clearly.

---

**Good luck with your interview!**

<a name="read-code-15"></a>

**Cleanup Cell — Close the Database Connection**

This cell closes the connection to the SQLite database. Always close database connections when finished to release system resources and ensure all pending writes are flushed to disk.

<nav aria-label="Code cell 15 navigation">
<a href="#skip-code-15" aria-label="Skip code cell 15 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# Clean up
conn.close()
print("Database connection closed")

<a name="skip-code-15"></a>

**Expected output:** `Database connection closed`

You have reached the end of the notebook. Save your work before submitting. Review the Reflection Questions section as a final interview preparation step.

<nav aria-label="Return navigation for code cell 15">
<a href="#read-code-15" aria-label="Go back and read code cell 15">&#8617; Go back and read the code cell</a>
</nav>